# Stage 0 — Amazon Reviews 2023 Data Preprocessing

<a href="https://colab.research.google.com/github/zixian0821-zoe/EN553744-final-project/blob/main/data_preprocessing/amazon_data_preprocess.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Code (canonical):** https://github.com/zixian0821-zoe/EN553744-final-project — full pipeline, training scripts, model definitions, and per-experiment runners.

**Course:** EN.553.744 Data Science for Large-Scale Graphs (Spring 2026, JHU AMS)
**Team:** Zixian Zhou · Yunwei Chai · Yang Song

## What this notebook does

Downloads the Amazon Reviews 2023 dataset (CDs/Vinyl + Books, 0core_rating_only) from Hugging Face Hub, merges the Books shards, materializes per-user × per-item interaction tables, and writes the cleaned parquet artifacts that Stage 1 (`pipeline/stage1_preprocess_graphs.py`) consumes.


In [ ]:
!pip -q install -U "datasets>=4.0.0" "huggingface_hub>=0.24.0" "fsspec>=2024.6.0" pyarrow pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 142.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas

In [ ]:
!ls -lh /content/*.parquet


ls: cannot access '/content/*.parquet': No such file or directory


In [ ]:
!rm -f /content/*.parquet
!mkdir -p /content/amazon_data


In [ ]:
!pip -q install -U huggingface_hub pandas pyarrow

from huggingface_hub import hf_hub_download
import os

REV = "6a62d7bf8b3f3a90943ac9c6ea800ae7736959ad"

files = [
    "0core_rating_only_CDs_and_Vinyl/full-00000-of-00001.parquet",
    "0core_rating_only_Books/full-00000-of-00005.parquet",
    "0core_rating_only_Books/full-00001-of-00005.parquet",
    "0core_rating_only_Books/full-00002-of-00005.parquet",
    "0core_rating_only_Books/full-00003-of-00005.parquet",
    "0core_rating_only_Books/full-00004-of-00005.parquet",
]

local_paths = []
for f in files:
    p = hf_hub_download(
        repo_id="McAuley-Lab/Amazon-Reviews-2023",
        repo_type="dataset",
        filename=f,
        revision=REV,
        local_dir="/content/amazon_data",
        local_dir_use_symlinks=False,
    )
    local_paths.append(p)
    print("Downloaded:", p, "size =", os.path.getsize(p))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


0core_rating_only_CDs_and_Vinyl/full-000(…):   0%|          | 0.00/155M [00:00<?, ?B/s]

Downloaded: /content/amazon_data/0core_rating_only_CDs_and_Vinyl/full-00000-of-00001.parquet size = 155313128


0core_rating_only_Books/full-00000-of-00(…):   0%|          | 0.00/171M [00:00<?, ?B/s]

Downloaded: /content/amazon_data/0core_rating_only_Books/full-00000-of-00005.parquet size = 171329938


...

Downloaded: /content/amazon_data/0core_rating_only_Books/full-00002-of-00005.parquet size = 192170925


0core_rating_only_Books/full-00003-of-00(…):   0%|          | 0.00/207M [00:00<?, ?B/s]

Downloaded: /content/amazon_data/0core_rating_only_Books/full-00003-of-00005.parquet size = 207223924


0core_rating_only_Books/full-00004-of-00(…):   0%|          | 0.00/234M [00:00<?, ?B/s]

Downloaded: /content/amazon_data/0core_rating_only_Books/full-00004-of-00005.parquet size = 234358929


In [ ]:
import os, glob

for f in glob.glob("/content/amazon_data/**/*.parquet", recursive=True):
    print(f, os.path.getsize(f))


/content/amazon_data/0core_rating_only_Books/full-00004-of-00005.parquet 234358929
/content/amazon_data/0core_rating_only_Books/full-00000-of-00005.parquet 171329938
/content/amazon_data/0core_rating_only_Books/full-00003-of-00005.parquet 207223924
/content/amazon_data/0core_rating_only_Books/full-00001-of-00005.parquet 186824880
/content/amazon_data/0core_rating_only_Books/full-00002-of-00005.parquet 192170925
/content/amazon_data/0core_rating_only_CDs_and_Vinyl/full-00000-of-00001.parquet 155313128


In [ ]:
import pandas as pd

cds = pd.read_parquet("/content/amazon_data/0core_rating_only_CDs_and_Vinyl/full-00000-of-00001.parquet")

books = pd.concat([
    pd.read_parquet("/content/amazon_data/0core_rating_only_Books/full-00000-of-00005.parquet"),
    pd.read_parquet("/content/amazon_data/0core_rating_only_Books/full-00001-of-00005.parquet"),
    pd.read_parquet("/content/amazon_data/0core_rating_only_Books/full-00002-of-00005.parquet"),
    pd.read_parquet("/content/amazon_data/0core_rating_only_Books/full-00003-of-00005.parquet"),
    pd.read_parquet("/content/amazon_data/0core_rating_only_Books/full-00004-of-00005.parquet"),
], ignore_index=True)

print("CDs shape:", cds.shape)
print("Books shape:", books.shape)
print("CDs columns:", cds.columns.tolist())
print("Books columns:", books.columns.tolist())


CDs shape: (4772071, 4)
Books shape: (29139329, 4)
CDs columns: ['user_id', 'parent_asin', 'rating', 'timestamp']
Books columns: ['user_id', 'parent_asin', 'rating', 'timestamp']


In [ ]:
import os
import numpy as np
import pandas as pd

USER_THRESHOLD_BOTH = 10

MIN_TARGET_ITEM_FREQ = 2

MIN_TARGET_INTER_PER_USER_FOR_SPLIT = 3

SOURCE_NAME = "CDs_and_Vinyl"
TARGET_NAME = "Books"

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
RANDOM_SEED = 42

OUTPUT_DIR = "/content/amazon_cd_book_thr10_targetItem2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.random.seed(RANDOM_SEED)

REQUIRED_COLS = ["user_id", "parent_asin", "rating", "timestamp"]

source_df = cds.copy()
target_df = books.copy()

print("Initial source shape:", source_df.shape)
print("Initial target shape:", target_df.shape)

for col in REQUIRED_COLS:
    print(f"{col}: source={col in source_df.columns}, target={col in target_df.columns}")

source_df = source_df[REQUIRED_COLS].dropna(subset=["user_id", "parent_asin", "timestamp"]).copy()
target_df = target_df[REQUIRED_COLS].dropna(subset=["user_id", "parent_asin", "timestamp"]).copy()

source_df["timestamp"] = pd.to_numeric(source_df["timestamp"], errors="coerce")
target_df["timestamp"] = pd.to_numeric(target_df["timestamp"], errors="coerce")

source_df = source_df.dropna(subset=["timestamp"]).copy()
target_df = target_df.dropna(subset=["timestamp"]).copy()

print("\nAfter basic cleaning:")
print("Source shape:", source_df.shape)
print("Target shape:", target_df.shape)

print("\nDeduplicating user-item pairs by latest timestamp...")

source_df = (
    source_df.sort_values(["user_id", "parent_asin", "timestamp"], kind="mergesort")
             .drop_duplicates(subset=["user_id", "parent_asin"], keep="last")
             .reset_index(drop=True)
)

target_df = (
    target_df.sort_values(["user_id", "parent_asin", "timestamp"], kind="mergesort")
             .drop_duplicates(subset=["user_id", "parent_asin"], keep="last")
             .reset_index(drop=True)
)

print("After dedup:")
print("Source shape:", source_df.shape)
print("Target shape:", target_df.shape)

print("\nFinding overlap users...")

source_users = set(source_df["user_id"].unique())
target_users = set(target_df["user_id"].unique())
overlap_users = source_users & target_users

source_df = source_df[source_df["user_id"].isin(overlap_users)].copy()
target_df = target_df[target_df["user_id"].isin(overlap_users)].copy()

print("Overlap users:", len(overlap_users))
print("Source shape after overlap filter:", source_df.shape)
print("Target shape after overlap filter:", target_df.shape)

print("\nApplying user threshold in both domains...")

source_user_counts = source_df["user_id"].value_counts()
target_user_counts = target_df["user_id"].value_counts()

qualified_users = (
    set(source_user_counts[source_user_counts >= USER_THRESHOLD_BOTH].index)
    & set(target_user_counts[target_user_counts >= USER_THRESHOLD_BOTH].index)
)

source_df = source_df[source_df["user_id"].isin(qualified_users)].copy()
target_df = target_df[target_df["user_id"].isin(qualified_users)].copy()

print("Qualified users:", len(qualified_users))
print("Source shape after user threshold:", source_df.shape)
print("Target shape after user threshold:", target_df.shape)

if source_df.empty or target_df.empty:
    raise ValueError("Data became empty right after user-threshold filtering.")

print(f"\nApplying mild one-pass target item filter: item freq >= {MIN_TARGET_ITEM_FREQ}")

target_item_counts = target_df["parent_asin"].value_counts()
keep_target_items = set(target_item_counts[target_item_counts >= MIN_TARGET_ITEM_FREQ].index)

target_df = target_df[target_df["parent_asin"].isin(keep_target_items)].copy()

print("Target shape after mild target-item filter:", target_df.shape)
print("Target unique items after filter:", target_df["parent_asin"].nunique())

target_user_counts_after = target_df["user_id"].value_counts()
split_capable_users = set(
    target_user_counts_after[target_user_counts_after >= MIN_TARGET_INTER_PER_USER_FOR_SPLIT].index
)

final_users = set(source_df["user_id"].unique()) & split_capable_users

source_df = source_df[source_df["user_id"].isin(final_users)].copy()
target_df = target_df[target_df["user_id"].isin(final_users)].copy()

print("\nAfter keeping split-capable users:")
print("Final users:", len(final_users))
print("Source shape:", source_df.shape)
print("Target shape:", target_df.shape)

if source_df.empty or target_df.empty:
    raise ValueError(
        "Data became empty after mild target-item filtering. "
        "Try MIN_TARGET_ITEM_FREQ = 1."
    )

print("\nCreating target train/val/test split...")

final_users = sorted(set(source_df["user_id"].unique()) & set(target_df["user_id"].unique()))
user2id = {u: i for i, u in enumerate(final_users)}

source_df["user_idx"] = source_df["user_id"].map(user2id)
target_df["user_idx"] = target_df["user_id"].map(user2id)

target_df = target_df.sort_values(["user_idx", "timestamp"]).reset_index(drop=True)

train_parts = []
val_parts = []
test_parts = []

for user_idx, grp in target_df.groupby("user_idx", sort=False):
    grp = grp.sort_values("timestamp").reset_index(drop=True)
    n = len(grp)

    if n < 3:
        train_parts.append(grp.copy())
        continue

    n_train = int(np.floor(n * TRAIN_RATIO))
    n_val = int(np.floor(n * VAL_RATIO))
    n_test = n - n_train - n_val

    if n_val == 0:
        n_val = 1
        n_train -= 1
    if n_test == 0:
        n_test = 1
        n_train -= 1
    if n_train <= 0:
        n_train = max(1, n - 2)
        n_val = 1
        n_test = n - n_train - n_val

    train_grp = grp.iloc[:n_train].copy()
    val_grp = grp.iloc[n_train:n_train + n_val].copy()
    test_grp = grp.iloc[n_train + n_val:].copy()

    if len(train_grp) > 0:
        train_parts.append(train_grp)
    if len(val_grp) > 0:
        val_parts.append(val_grp)
    if len(test_grp) > 0:
        test_parts.append(test_grp)

train_df = pd.concat(train_parts, ignore_index=True)
val_df = pd.concat(val_parts, ignore_index=True) if len(val_parts) > 0 else target_df.iloc[0:0].copy()
test_df = pd.concat(test_parts, ignore_index=True) if len(test_parts) > 0 else target_df.iloc[0:0].copy()

print("Before seen-item filter:")
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

train_target_items = set(train_df["parent_asin"].unique())

val_df = val_df[val_df["parent_asin"].isin(train_target_items)].copy()
test_df = test_df[test_df["parent_asin"].isin(train_target_items)].copy()

target_df = target_df[target_df["parent_asin"].isin(train_target_items)].copy()

print("\nAfter seen-item filter:")
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print("Target interactions kept:", len(target_df))
print("Target items kept:", target_df["parent_asin"].nunique())

print("\nCreating integer ID mappings...")

source_items = sorted(source_df["parent_asin"].unique())
target_items = sorted(train_target_items)

source_item2id = {it: i for i, it in enumerate(source_items)}
target_item2id = {it: i for i, it in enumerate(target_items)}

source_df["item_idx"] = source_df["parent_asin"].map(source_item2id)
target_df["item_idx"] = target_df["parent_asin"].map(target_item2id)
train_df["item_idx"] = train_df["parent_asin"].map(target_item2id)
val_df["item_idx"] = val_df["parent_asin"].map(target_item2id)
test_df["item_idx"] = test_df["parent_asin"].map(target_item2id)

user_mapping = pd.DataFrame({
    "user_id": final_users,
    "user_idx": [user2id[u] for u in final_users]
})

source_item_mapping = pd.DataFrame({
    "parent_asin": source_items,
    "source_item_idx": [source_item2id[it] for it in source_items]
})

target_item_mapping = pd.DataFrame({
    "parent_asin": target_items,
    "target_item_idx": [target_item2id[it] for it in target_items]
})

print("\nSaving outputs...")

source_out = source_df[["user_id", "parent_asin", "rating", "timestamp", "user_idx", "item_idx"]].copy()
target_out = target_df[["user_id", "parent_asin", "rating", "timestamp", "user_idx", "item_idx"]].copy()
train_out = train_df[["user_id", "parent_asin", "rating", "timestamp", "user_idx", "item_idx"]].copy()
val_out = val_df[["user_id", "parent_asin", "rating", "timestamp", "user_idx", "item_idx"]].copy()
test_out = test_df[["user_id", "parent_asin", "rating", "timestamp", "user_idx", "item_idx"]].copy()

source_out.to_csv(os.path.join(OUTPUT_DIR, "source_interactions.csv"), index=False)
target_out.to_csv(os.path.join(OUTPUT_DIR, "target_interactions.csv"), index=False)
train_out.to_csv(os.path.join(OUTPUT_DIR, "train.csv"), index=False)
val_out.to_csv(os.path.join(OUTPUT_DIR, "val.csv"), index=False)
test_out.to_csv(os.path.join(OUTPUT_DIR, "test.csv"), index=False)

user_mapping.to_csv(os.path.join(OUTPUT_DIR, "user_mapping.csv"), index=False)
source_item_mapping.to_csv(os.path.join(OUTPUT_DIR, "source_item_mapping.csv"), index=False)
target_item_mapping.to_csv(os.path.join(OUTPUT_DIR, "target_item_mapping.csv"), index=False)

summary_df = pd.DataFrame([{
    "source_domain": SOURCE_NAME,
    "target_domain": TARGET_NAME,
    "user_threshold_both_domains": USER_THRESHOLD_BOTH,
    "min_target_item_freq": MIN_TARGET_ITEM_FREQ,
    "n_users": len(final_users),
    "source_interactions": len(source_out),
    "target_interactions": len(target_out),
    "source_items": len(source_items),
    "target_items": len(target_items),
    "train_size": len(train_out),
    "val_size": len(val_out),
    "test_size": len(test_out),
    "avg_source_inter_per_user": round(len(source_out) / max(len(final_users), 1), 4),
    "avg_target_inter_per_user": round(len(target_out) / max(len(final_users), 1), 4),
}])

summary_df.to_csv(os.path.join(OUTPUT_DIR, "summary.csv"), index=False)

print("Saved to:", OUTPUT_DIR)
print("\n=== Summary ===")
display(summary_df)

print("\nFiles generated:")
for fn in sorted(os.listdir(OUTPUT_DIR)):
    print(fn)


Initial source shape: (4772071, 4)
Initial target shape: (29139329, 4)
user_id: source=True, target=True
parent_asin: source=True, target=True
rating: source=True, target=True
timestamp: source=True, target=True

After basic cleaning:
Source shape: (4772071, 4)
Target shape: (29139329, 4)

Deduplicating user-item pairs by latest timestamp...
After dedup:
Source shape: (4772071, 4)
Target shape: (29139329, 4)

Finding overlap users...
Overlap users: 877324
Source shape after overlap filter: (2993771, 4)
Target shape after overlap filter: (5616579, 4)

Applying user threshold in both domains...
Qualified users: 16376
Source shape after user threshold: (571685, 4)
Target shape after user threshold: (711872, 4)

Applying mild one-pass target item filter: item freq >= 2
Target shape after mild target-item filter: (362609, 4)
Target unique items after filter: 108256

After keeping split-capable users:
Final users: 15824
Source shape: (555963, 4)
Target shape: (361767, 4)

Creating target tra

,source_domain,target_domain,user_threshold_both_domains,min_target_item_freq,n_users,source_interactions,target_interactions,source_items,target_items,train_size,val_size,test_size,avg_source_inter_per_user,avg_target_inter_per_user
0,CDs_and_Vinyl,Books,10,2,15824,555963,347212,236399,101723,276919,30012,40281,35.1342,21.9421



Files generated:
source_interactions.csv
source_item_mapping.csv
summary.csv
target_interactions.csv
target_item_mapping.csv
test.csv
train.csv
user_mapping.csv
val.csv
